# Eksperimen CNN Embedding + ANFIS Optimized — 2 GPU T4 Max Batch

Notebook ini tetap menjadikan **ANFIS sebagai sistem soft-computing utama** untuk klasifikasi akhir. Bagian yang dioptimalkan adalah representasi fitur citra dan pencarian konfigurasi, bukan mengganti ANFIS dengan classifier deep learning.

Pipeline:

```text
COCO JSON + citra UAV
-> grid 128x128 + label dari mask COCO
-> CNN pretrained feature extractor dengan dukungan multi-GPU untuk GPU T4 x2
-> auto-tune batch terbesar yang lolos memori GPU
-> embedding citra original saja tanpa flip-averaging
-> PCA + MinMax scaling
-> ANFIS final
-> evaluasi Aman / Tersebar / Kritis
```

Fitur manual seperti GLCM, ExG, warna, edge, dan `dataset_fitur_sampah.csv` tidak dipakai sebagai input ataupun sumber label.


## 1. Import dan Konfigurasi Path

Notebook bisa dijalankan dari root project atau dari folder `notebooks/`.


In [1]:
!pip install pycocotools opencv-python tqdm joblib seaborn scikit-learn


In [2]:
from pathlib import Path
from PIL import Image, ImageOps
import copy
import json
import math
import os
import random
import warnings

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from pycocotools.coco import COCO

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau

from torchvision import transforms
from torchvision.models import (
    efficientnet_b0, EfficientNet_B0_Weights,
    resnet18, ResNet18_Weights,
    resnet50, ResNet50_Weights,
)

warnings.filterwarnings('ignore')


In [3]:
def find_kaggle_input_root():
    candidates = [
        Path('/kaggle/input/datasets/dzikribassyril/dronewate-data'),
        Path('/kaggle/input/dronewate-data'),
        Path('/kaggle/input/dronewaste-data'),
        Path('/kaggle/input/dronewaste-anfis-project'),
    ]
    for candidate in candidates:
        if (candidate / 'data' / 'raw' / 'annotations' / 'dronewaste_v2.0.json').exists():
            return candidate

    input_dir = Path('/kaggle/input')
    if input_dir.exists():
        matches = list(input_dir.rglob('dronewaste_v2.0.json'))
        if matches:
            return matches[0].parents[3]
    return None

if Path('/kaggle').exists():
    KAGGLE_MODE = True
    PROJECT_ROOT = Path('/kaggle')
    INPUT_ROOT = find_kaggle_input_root()
    if INPUT_ROOT is None:
        raise FileNotFoundError('Tidak menemukan dronewaste_v2.0.json di /kaggle/input. Pastikan Kaggle Dataset sudah di-attach.')
    WORKING_ROOT = Path('/kaggle/working')
else:
    KAGGLE_MODE = False
    CURRENT_DIR = Path.cwd()
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
    INPUT_ROOT = PROJECT_ROOT
    WORKING_ROOT = PROJECT_ROOT

DATA_DIR = INPUT_ROOT / 'data'
RAW_DATA_DIR = DATA_DIR / 'raw'
IMAGE_DIR = RAW_DATA_DIR / 'images'
ANNOTATION_JSON = RAW_DATA_DIR / 'annotations' / 'dronewaste_v2.0.json'

OUTPUT_DATA_DIR = WORKING_ROOT / 'data'
PROCESSED_DATA_DIR = OUTPUT_DATA_DIR / 'processed'
MODEL_DIR = WORKING_ROOT / 'models'

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('Kaggle mode:', KAGGLE_MODE)
print('Input root:', INPUT_ROOT)
print('Working root:', WORKING_ROOT)
print('Image dir:', IMAGE_DIR)
print('Annotation JSON:', ANNOTATION_JSON)


Kaggle mode: True
Input root: /kaggle/input/datasets/dzikribassyril/dronewate-data
Working root: /kaggle/working
Image dir: /kaggle/input/datasets/dzikribassyril/dronewate-data/data/raw/images
Annotation JSON: /kaggle/input/datasets/dzikribassyril/dronewate-data/data/raw/annotations/dronewaste_v2.0.json


In [4]:
SEED = 42
GRID_SIZE = 128
NUM_CLASSES = 3

# Backbone lebih kuat daripada ResNet18 frozen. Pilihan: efficientnet_b0, resnet50, resnet18
BACKBONE_NAME = 'efficientnet_b0'

# Versi cepat: hanya memakai grid original, tanpa hflip/vflip saat ekstraksi embedding.
# Ini mengurangi forward pass CNN dari 3x menjadi 1x.
EMBEDDING_AUGMENTATIONS = ['original']

# Mode hemat GPU: tetap fine-tune CNN agar embedding adaptif, tetapi hanya singkat.
# Classifier CNN hanya bantu belajar embedding; klasifikasi final tetap memakai ANFIS.
TRAIN_SUPERVISED_BACKBONE = True
FORCE_REFIT_BACKBONE = False
FAST_GPU_BUDGET_MODE = True
FINE_TUNE_EPOCHS = 5
FINE_TUNE_PATIENCE = 2
FINE_TUNE_LR = 2e-4
FINE_TUNE_WEIGHT_DECAY = 1e-4
FINE_TUNE_LABEL_SMOOTHING = 0.05
FINE_TUNE_UNFREEZE_LAST_BLOCKS = 1

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
GPU_IDS = list(range(GPU_COUNT))
USE_MULTI_GPU = DEVICE.type == 'cuda' and GPU_COUNT > 1

# Optimasi untuk Kaggle GPU T4 x2.
# Batch dibuat agresif, lalu dites otomatis dengan dummy forward/backward.
# Jika OOM, notebook otomatis turun ke kandidat batch berikutnya.
AUTO_TUNE_CNN_BATCH = True
BATCH_TUNE_INPUT_SIZE = 224

if GPU_COUNT >= 2:
    # Global batch; DataParallel membagi batch ke semua GPU.
    # Pada T4 x2: 512 ~= 256/GPU untuk fine-tuning, 2048 ~= 1024/GPU untuk embedding no-grad.
    BATCH_SIZE_FINETUNE_CANDIDATES = [512, 384, 256, 192, 128, 96, 64]
    BATCH_SIZE_EMBEDDING_CANDIDATES = [2048, 1536, 1024, 768, 512, 384, 256, 128]
elif GPU_COUNT == 1:
    BATCH_SIZE_FINETUNE_CANDIDATES = [256, 192, 128, 96, 64, 32]
    BATCH_SIZE_EMBEDDING_CANDIDATES = [1024, 768, 512, 384, 256, 128, 64]
else:
    BATCH_SIZE_FINETUNE_CANDIDATES = [32]
    BATCH_SIZE_EMBEDDING_CANDIDATES = [64]

BATCH_SIZE_FINETUNE = BATCH_SIZE_FINETUNE_CANDIDATES[0]
BATCH_SIZE_EMBEDDING = BATCH_SIZE_EMBEDDING_CANDIDATES[0]
BATCH_SIZE_TRAIN = 2048  # Batch ANFIS, aman karena fitur PCA kecil dan berjalan di cuda:0.
PREDICT_BATCH_SIZE = 8192

# DataLoader dibuat konservatif untuk Kaggle. persistent_workers/prefetch tinggi kadang
# menggantung antar-epoch pada dataset PIL crop, jadi timeout dipakai agar tidak silent 12 jam.
CPU_COUNT = os.cpu_count() or 2
NUM_WORKERS = min(2, CPU_COUNT) if KAGGLE_MODE else 0
PIN_MEMORY = torch.cuda.is_available()
PERSISTENT_WORKERS = False
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None
DATALOADER_TIMEOUT = 180 if NUM_WORKERS > 0 else 0

N_SPLITS_TEST = 5
N_SPLITS_VAL = 5
FORCE_REBUILD_GRID_METADATA = False
FORCE_REEXTRACT_EMBEDDING = False

RUN_GA_SEARCH = True
GA_POPULATION = 4
GA_GENERATIONS = 2
GA_MUTATION_RATE = 0.35
SEARCH_EPOCHS = 70
SEARCH_PATIENCE = 10
FINAL_EPOCHS = 220
FINAL_PATIENCE = 30

SEARCH_SPACE = {
    'pca_components': [24, 30, 36],
    'num_rules': [10, 12, 15],
    'learning_rate': [0.003, 0.0015],
    'weight_decay': [1e-5, 5e-5],
    'class_weight_power': [1.0, 1.2],
    'label_smoothing': [0.02, 0.05],
}

FALLBACK_CONFIG = {
    'pca_components': 24,
    'num_rules': 12,
    'learning_rate': 0.003,
    'weight_decay': 1e-5,
    'class_weight_power': 1.0,
    'label_smoothing': 0.02,
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')


def maybe_wrap_multi_gpu(model, name='model'):
    """Wrap model dengan DataParallel jika tersedia lebih dari 1 GPU."""
    if USE_MULTI_GPU:
        print(f'Menggunakan DataParallel untuk {name}: GPU {GPU_IDS}')
        return nn.DataParallel(model, device_ids=GPU_IDS)
    return model


def unwrap_model(model):
    """Ambil model asli dari DataParallel agar state_dict bersih saat save/load."""
    return model.module if isinstance(model, nn.DataParallel) else model


def strip_module_prefix(state_dict):
    """Kompatibel dengan checkpoint yang disimpan dari DataParallel maupun single-GPU."""
    if not any(k.startswith('module.') for k in state_dict.keys()):
        return state_dict
    return {k.replace('module.', '', 1): v for k, v in state_dict.items()}


def fast_loader_kwargs():
    """Argumen DataLoader yang aman untuk CPU worker 0 maupun >0."""
    kwargs = {
        'num_workers': NUM_WORKERS,
        'pin_memory': PIN_MEMORY,
    }
    if NUM_WORKERS > 0:
        kwargs['persistent_workers'] = PERSISTENT_WORKERS
        kwargs['prefetch_factor'] = PREFETCH_FACTOR
        kwargs['timeout'] = DATALOADER_TIMEOUT
    return kwargs


def _is_cuda_oom(error):
    msg = str(error).lower()
    return isinstance(error, RuntimeError) and ('out of memory' in msg or 'cuda error: out of memory' in msg)


def tune_cnn_batch_size(model, candidate_batches, train_mode=False, name='CNN'):
    """Cari batch terbesar dari daftar kandidat yang lolos tes memori GPU.

    Tes ini hanya memakai dummy tensor berukuran sama dengan input pretrained torchvision (224x224).
    Jika batch terbesar OOM, notebook otomatis mencoba batch lebih kecil.
    """
    if not AUTO_TUNE_CNN_BATCH or DEVICE.type != 'cuda':
        return candidate_batches[0]

    old_training_state = model.training
    params = [p for p in model.parameters() if p.requires_grad]
    if train_mode and len(params) == 0:
        train_mode = False

    print(f'Auto-tune batch {name}. Kandidat:', candidate_batches)
    for bsz in candidate_batches:
        try:
            torch.cuda.empty_cache()
            x = torch.randn(bsz, 3, BATCH_TUNE_INPUT_SIZE, BATCH_TUNE_INPUT_SIZE, device=DEVICE)

            if train_mode:
                model.train()
                y = torch.randint(0, NUM_CLASSES, (bsz,), device=DEVICE)
                optimizer_probe = optim.SGD(params, lr=1e-5)
                optimizer_probe.zero_grad(set_to_none=True)
                with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=True):
                    logits = model(x)
                    loss = F.cross_entropy(logits, y)
                loss.backward()
                optimizer_probe.zero_grad(set_to_none=True)
                del optimizer_probe, loss, logits, y
            else:
                model.eval()
                with torch.no_grad():
                    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=True):
                        feats = model(x)
                del feats

            del x
            torch.cuda.empty_cache()
            model.train(old_training_state)
            print(f'Batch {name} dipilih: {bsz}')
            return bsz
        except RuntimeError as err:
            if _is_cuda_oom(err):
                print(f'Batch {name} {bsz} OOM, turun ke kandidat berikutnya.')
                try:
                    del x
                except Exception:
                    pass
                torch.cuda.empty_cache()
                continue
            model.train(old_training_state)
            raise

    fallback = candidate_batches[-1]
    torch.cuda.empty_cache()
    model.train(old_training_state)
    print(f'Semua kandidat besar OOM. Memakai fallback batch {name}: {fallback}')
    return fallback


print('Device utama:', DEVICE)
print('CUDA tersedia:', torch.cuda.is_available())
print('Jumlah GPU:', GPU_COUNT)
if DEVICE.type == 'cuda':
    for gpu_id in GPU_IDS:
        print(f'GPU {gpu_id}:', torch.cuda.get_device_name(gpu_id))
print('Multi-GPU aktif:', USE_MULTI_GPU)
print('Fast GPU budget mode:', FAST_GPU_BUDGET_MODE)
print('Auto tune batch:', AUTO_TUNE_CNN_BATCH)
print('Kandidat batch fine-tune:', BATCH_SIZE_FINETUNE_CANDIDATES)
print('Kandidat batch embedding:', BATCH_SIZE_EMBEDDING_CANDIDATES)
print('Batch ANFIS:', BATCH_SIZE_TRAIN)
print('Predict batch:', PREDICT_BATCH_SIZE)
print('CPU count:', CPU_COUNT)
print('Num workers:', NUM_WORKERS)
print('Pin memory:', PIN_MEMORY)
print('Persistent workers:', PERSISTENT_WORKERS)
print('Prefetch factor:', PREFETCH_FACTOR)
print('DataLoader timeout:', DATALOADER_TIMEOUT)
print('Augmentasi embedding:', EMBEDDING_AUGMENTATIONS)


Device utama: cuda:0
CUDA tersedia: True
Jumlah GPU: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
Multi-GPU aktif: True
Auto tune batch: True
Kandidat batch fine-tune: [512, 384, 256, 192, 128, 96, 64]
Kandidat batch embedding: [2048, 1536, 1024, 768, 512, 384, 256, 128]
Batch ANFIS: 2048
Predict batch: 8192
CPU count: 4
Num workers: 4
Pin memory: True
Persistent workers: True
Prefetch factor: 4
Augmentasi embedding: ['original']


In [5]:
AUG_TAG = '-'.join(EMBEDDING_AUGMENTATIONS)
FT_TAG = 'finetuned' if TRAIN_SUPERVISED_BACKBONE else 'frozen'
CACHE_PREFIX = f'{BACKBONE_NAME}_{FT_TAG}_{AUG_TAG}_grid{GRID_SIZE}'

GRID_METADATA_CSV = PROCESSED_DATA_DIR / 'grid_metadata_coco.csv'
EMBEDDING_NPY = PROCESSED_DATA_DIR / f'embeddings_{CACHE_PREFIX}.npy'
EMBEDDING_META_CSV = PROCESSED_DATA_DIR / f'embeddings_{CACHE_PREFIX}_metadata.csv'
CNN_FEATURE_CSV = PROCESSED_DATA_DIR / f'features_{CACHE_PREFIX}_best_pca_anfis.csv'

PCA_PATH = MODEL_DIR / f'pca_{CACHE_PREFIX}_best.pkl'
SCALER_PATH = MODEL_DIR / f'scaler_{CACHE_PREFIX}_best.pkl'
MODEL_PATH = MODEL_DIR / f'model_{CACHE_PREFIX}_anfis.pth'
SEARCH_RESULTS_CSV = PROCESSED_DATA_DIR / f'ga_search_{CACHE_PREFIX}.csv'
FINE_TUNED_BACKBONE_PATH = MODEL_DIR / f'finetuned_{BACKBONE_NAME}_grid{GRID_SIZE}_classifier.pth'
FINE_TUNED_BACKBONE_CKPT_PATH = MODEL_DIR / f'finetuned_{BACKBONE_NAME}_grid{GRID_SIZE}_classifier_checkpoint.pth'

print('Grid metadata:', GRID_METADATA_CSV)
print('Embedding cache:', EMBEDDING_NPY)
print('Feature CSV:', CNN_FEATURE_CSV)
print('Model path:', MODEL_PATH)
print('Fine-tuned CNN path:', FINE_TUNED_BACKBONE_PATH)
print('Fine-tuned CNN checkpoint:', FINE_TUNED_BACKBONE_CKPT_PATH)


Grid metadata: /kaggle/working/data/processed/grid_metadata_coco.csv
Embedding cache: /kaggle/working/data/processed/embeddings_efficientnet_b0_finetuned_original_grid128.npy
Feature CSV: /kaggle/working/data/processed/features_efficientnet_b0_finetuned_original_grid128_best_pca_anfis.csv
Model path: /kaggle/working/models/model_efficientnet_b0_finetuned_original_grid128_anfis.pth
Fine-tuned CNN path: /kaggle/working/models/finetuned_efficientnet_b0_grid128_classifier.pth


## 2. Bangun Grid dan Label dari COCO

Grid metadata dibuat langsung dari anotasi COCO dan citra asli. Mask ground truth hanya dipakai untuk membuat label training, bukan sebagai fitur input model.

Aturan label:

```text
Kelas 0 Aman      : kepadatan < 0.05
Kelas 1 Tersebar  : 0.05 <= kepadatan <= 0.40
Kelas 2 Kritis    : kepadatan > 0.40
```


In [6]:
def generate_binary_mask(image_id, coco):
    img_info = coco.loadImgs(image_id)[0]
    height, width = img_info['height'], img_info['width']
    mask = np.zeros((height, width), dtype=np.uint8)

    ann_ids = coco.getAnnIds(imgIds=[image_id])
    anns = coco.loadAnns(ann_ids)

    for ann in anns:
        ann_mask = coco.annToMask(ann)
        mask[ann_mask > 0] = 255

    return mask


def density_to_label(density):
    if density < 0.05:
        return 0
    if density <= 0.40:
        return 1
    return 2


def build_grid_metadata_from_coco(coco, image_dir, grid_size=128):
    rows = []
    image_ids = coco.getImgIds()

    for image_id in tqdm(image_ids, desc="Building grid labels from COCO", unit="image", dynamic_ncols=True):
        img_info = coco.loadImgs(image_id)[0]
        image_file = img_info['file_name']
        img_path = Path(image_dir) / image_file

        if not img_path.exists():
            raise FileNotFoundError(f"File gambar tidak ditemukan: {img_path}")

        img = cv2.imread(str(img_path))
        if img is None:
            raise FileNotFoundError(f"Gambar tidak bisa dibaca: {img_path}")

        height, width = img.shape[:2]
        mask = generate_binary_mask(image_id, coco)

        if mask.shape[:2] != (height, width):
            raise ValueError(f"Ukuran mask dan gambar tidak sama untuk {image_file}: mask={mask.shape}, image={img.shape}")

        for y in range(0, height, grid_size):
            for x in range(0, width, grid_size):
                grid_mask = mask[y:y + grid_size, x:x + grid_size]

                if grid_mask.shape[:2] != (grid_size, grid_size):
                    continue

                density = float(np.mean(grid_mask > 0))
                y_target = density_to_label(density)

                rows.append({
                    'Grid_ID': f"{image_file}_X{x}_Y{y}",
                    'image_id': image_id,
                    'image_file': image_file,
                    'x': x,
                    'y': y,
                    'kepadatan': density,
                    'Y_Target': y_target,
                })

    return pd.DataFrame(rows)


if GRID_METADATA_CSV.exists() and not FORCE_REBUILD_GRID_METADATA:
    print("Menemukan grid metadata COCO yang sudah ada. Memuat dari:", GRID_METADATA_CSV)
    df_grid = pd.read_csv(GRID_METADATA_CSV)
else:
    coco_api = COCO(str(ANNOTATION_JSON))
    df_grid = build_grid_metadata_from_coco(coco_api, IMAGE_DIR, GRID_SIZE)
    df_grid.to_csv(GRID_METADATA_CSV, index=False)
    print("Grid metadata COCO disimpan ke:", GRID_METADATA_CSV)

required_cols = {'Grid_ID', 'image_file', 'x', 'y', 'kepadatan', 'Y_Target'}
missing_cols = required_cols - set(df_grid.columns)
if missing_cols:
    raise ValueError(f"Grid metadata tidak lengkap. Kolom hilang: {sorted(missing_cols)}")

missing_files = sorted(set(df_grid['image_file']) - {p.name for p in IMAGE_DIR.glob('*.png')})
if missing_files:
    raise FileNotFoundError(f"Ada {len(missing_files)} gambar di metadata tetapi tidak ada di folder images. Contoh: {missing_files[:5]}")

print("Total grid:", len(df_grid))
print("Total image unik:", df_grid['image_file'].nunique())
print("Distribusi kelas:")
print(df_grid['Y_Target'].value_counts().sort_index())
print("Kepadatan per kelas:")
print(df_grid.groupby('Y_Target')['kepadatan'].describe()[['count', 'mean', 'min', 'max']].round(4))

df_grid[['Grid_ID', 'image_file', 'x', 'y', 'kepadatan', 'Y_Target']].head()


loading annotations into memory...
Done (t=0.13s)
creating index...
index created!


Building grid labels from COCO:   0%|          | 0/1171 [00:00<?, ?image/s]

Grid metadata COCO disimpan ke: /kaggle/working/data/processed/grid_metadata_coco.csv
Total grid: 29275
Total image unik: 1171
Distribusi kelas:
Y_Target
0    18355
1     3407
2     7513
Name: count, dtype: int64
Kepadatan per kelas:
            count    mean   min     max
Y_Target                               
0         18355.0  0.0014  0.00  0.0500
1          3407.0  0.2049  0.05  0.3999
2          7513.0  0.8458  0.40  1.0000


,Grid_ID,image_file,x,y,kepadatan,Y_Target
0,site3_26.png_X0_Y0,site3_26.png,0,0,0.0,0
1,site3_26.png_X128_Y0,site3_26.png,128,0,0.0,0
2,site3_26.png_X256_Y0,site3_26.png,256,0,0.0,0
3,site3_26.png_X384_Y0,site3_26.png,384,0,0.0,0
4,site3_26.png_X512_Y0,site3_26.png,512,0,0.0,0


## 3. Split Data Berbasis Gambar

Split dilakukan berdasarkan `image_file` agar grid dari citra yang sama tidak bocor ke train dan test sekaligus. Ini lebih ketat daripada random split per grid.


In [7]:
def make_group_splits(df, label_col='Y_Target', group_col='image_file'):
    y = df[label_col].values
    groups = df[group_col].values
    indices = np.arange(len(df))

    sgkf_test = StratifiedGroupKFold(n_splits=N_SPLITS_TEST, shuffle=True, random_state=SEED)
    trainval_idx, test_idx = next(sgkf_test.split(indices, y, groups))

    df_trainval = df.iloc[trainval_idx].reset_index(drop=False).rename(columns={'index': 'original_idx'})
    y_trainval = df_trainval[label_col].values
    groups_trainval = df_trainval[group_col].values
    indices_trainval = np.arange(len(df_trainval))

    sgkf_val = StratifiedGroupKFold(n_splits=N_SPLITS_VAL, shuffle=True, random_state=SEED + 1)
    train_rel_idx, val_rel_idx = next(sgkf_val.split(indices_trainval, y_trainval, groups_trainval))

    train_idx = df_trainval.iloc[train_rel_idx]['original_idx'].values
    val_idx = df_trainval.iloc[val_rel_idx]['original_idx'].values

    return train_idx, val_idx, test_idx

train_idx, val_idx, test_idx = make_group_splits(df_grid)

for name, idx in [('Train', train_idx), ('Val', val_idx), ('Test', test_idx)]:
    subset = df_grid.iloc[idx]
    print(f"{name}: rows={len(subset)}, images={subset['image_file'].nunique()}")
    print(subset['Y_Target'].value_counts(normalize=True).sort_index().round(3))
    print()


Train: rows=18725, images=749
Y_Target
0    0.618
1    0.116
2    0.267
Name: proportion, dtype: float64

Val: rows=4700, images=188
Y_Target
0    0.633
1    0.123
2    0.244
Name: proportion, dtype: float64

Test: rows=5850, images=234
Y_Target
0    0.652
1    0.114
2    0.234
Name: proportion, dtype: float64



## 4. CNN Feature Extractor

Default memakai EfficientNet-B0 pretrained. CNN hanya menjadi feature extractor; classifier akhir tetap ANFIS.


In [8]:
class EfficientNetFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.features = model.features
        self.avgpool = model.avgpool

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)


def build_cnn_feature_extractor(backbone_name):
    if backbone_name == 'efficientnet_b0':
        weights = EfficientNet_B0_Weights.DEFAULT
        model = efficientnet_b0(weights=weights)
        feature_extractor = EfficientNetFeatureExtractor(model)
        embedding_dim = model.classifier[1].in_features
        preprocess = weights.transforms()
    elif backbone_name == 'resnet50':
        weights = ResNet50_Weights.DEFAULT
        model = resnet50(weights=weights)
        feature_extractor = nn.Sequential(*list(model.children())[:-1], nn.Flatten())
        embedding_dim = model.fc.in_features
        preprocess = weights.transforms()
    elif backbone_name == 'resnet18':
        weights = ResNet18_Weights.DEFAULT
        model = resnet18(weights=weights)
        feature_extractor = nn.Sequential(*list(model.children())[:-1], nn.Flatten())
        embedding_dim = model.fc.in_features
        preprocess = weights.transforms()
    else:
        raise ValueError(f'Backbone tidak dikenal: {backbone_name}')

    feature_extractor = feature_extractor.to(DEVICE)
    feature_extractor = maybe_wrap_multi_gpu(feature_extractor, name=f'{backbone_name} feature extractor')
    feature_extractor.eval()
    return feature_extractor, preprocess, embedding_dim

cnn_feature_extractor, preprocess, EMBEDDING_DIM = build_cnn_feature_extractor(BACKBONE_NAME)
print('Backbone:', BACKBONE_NAME)
print('Embedding dim:', EMBEDDING_DIM)
print('Augmentations:', EMBEDDING_AUGMENTATIONS)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth



  0%|          | 0.00/20.5M [00:00<?, ?B/s]


 48%|████▊     | 9.75M/20.5M [00:00<00:00, 102MB/s]


100%|██████████| 20.5M/20.5M [00:00<00:00, 145MB/s]

Menggunakan DataParallel untuk efficientnet_b0 feature extractor: GPU [0, 1]
Backbone: efficientnet_b0
Embedding dim: 1280
Augmentations: ['original']


## 5. Supervised Fine-Tuning CNN untuk Embedding

Agar embedding lebih spesifik terhadap sampah UAV, CNN pretrained dilatih singkat memakai label grid. Head klasifikasi CNN ini hanya dipakai saat belajar fitur; setelah training, head dibuang dan embedding-nya diberikan ke ANFIS.


In [9]:
class SingleGridImageDataset(Dataset):
    def __init__(self, df, image_dir, transform):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.image_dir / row['image_file']
        img = cv2.imread(str(img_path))
        if img is None:
            raise FileNotFoundError(f'Gambar tidak bisa dibaca: {img_path}')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        grid = img[row['y']:row['y'] + GRID_SIZE, row['x']:row['x'] + GRID_SIZE]
        if grid.shape[:2] != (GRID_SIZE, GRID_SIZE):
            raise ValueError(f'Grid tidak valid pada {row["Grid_ID"]}: {grid.shape}')
        pil_grid = Image.fromarray(grid)
        return self.transform(pil_grid), int(row['Y_Target'])


def build_cnn_classifier(backbone_name):
    if backbone_name == 'efficientnet_b0':
        weights = EfficientNet_B0_Weights.DEFAULT
        model = efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
        preprocess = weights.transforms()
    elif backbone_name == 'resnet50':
        weights = ResNet50_Weights.DEFAULT
        model = resnet50(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, NUM_CLASSES)
        preprocess = weights.transforms()
    elif backbone_name == 'resnet18':
        weights = ResNet18_Weights.DEFAULT
        model = resnet18(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, NUM_CLASSES)
        preprocess = weights.transforms()
    else:
        raise ValueError(f'Backbone tidak dikenal: {backbone_name}')
    return model, preprocess, in_features


def unfreeze_for_finetune(model, backbone_name):
    for param in model.parameters():
        param.requires_grad = False

    if backbone_name == 'efficientnet_b0':
        for param in model.classifier.parameters():
            param.requires_grad = True
        for block in model.features[-FINE_TUNE_UNFREEZE_LAST_BLOCKS:]:
            for param in block.parameters():
                param.requires_grad = True
    else:
        for param in model.fc.parameters():
            param.requires_grad = True
        for param in model.layer4.parameters():
            param.requires_grad = True
        if FINE_TUNE_UNFREEZE_LAST_BLOCKS >= 2:
            for param in model.layer3.parameters():
                param.requires_grad = True


def feature_extractor_from_classifier(model, backbone_name):
    model = unwrap_model(model)
    if backbone_name == 'efficientnet_b0':
        return EfficientNetFeatureExtractor(model)
    return nn.Sequential(*list(model.children())[:-1], nn.Flatten())


def train_supervised_backbone():
    base_model, ft_preprocess, _ = build_cnn_classifier(BACKBONE_NAME)

    if FINE_TUNED_BACKBONE_PATH.exists() and not FORCE_REFIT_BACKBONE:
        print('Memuat fine-tuned CNN:', FINE_TUNED_BACKBONE_PATH)
        state = torch.load(FINE_TUNED_BACKBONE_PATH, map_location=DEVICE)
        base_model.load_state_dict(strip_module_prefix(state))
        base_model = base_model.to(DEVICE)
        return base_model, ft_preprocess

    unfreeze_for_finetune(base_model, BACKBONE_NAME)
    base_model = base_model.to(DEVICE)
    model = maybe_wrap_multi_gpu(base_model, name=f'{BACKBONE_NAME} classifier fine-tune')

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Trainable CNN params: {trainable:,}/{total:,}')

    global BATCH_SIZE_FINETUNE
    BATCH_SIZE_FINETUNE = tune_cnn_batch_size(
        model,
        BATCH_SIZE_FINETUNE_CANDIDATES,
        train_mode=True,
        name='fine-tune CNN',
    )
    print('Batch fine-tune efektif:', BATCH_SIZE_FINETUNE)

    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.12, hue=0.03),
        ft_preprocess,
    ])
    val_transform = ft_preprocess

    train_df = df_grid.iloc[train_idx].reset_index(drop=True)
    val_df = df_grid.iloc[val_idx].reset_index(drop=True)

    train_loader = DataLoader(
        SingleGridImageDataset(train_df, IMAGE_DIR, train_transform),
        batch_size=BATCH_SIZE_FINETUNE,
        shuffle=True,
        **fast_loader_kwargs(),
    )
    val_loader = DataLoader(
        SingleGridImageDataset(val_df, IMAGE_DIR, val_transform),
        batch_size=BATCH_SIZE_FINETUNE,
        shuffle=False,
        **fast_loader_kwargs(),
    )

    counts = np.bincount(train_df['Y_Target'].values, minlength=NUM_CLASSES)
    class_weights = torch.tensor([len(train_df) / (NUM_CLASSES * max(c, 1)) for c in counts], dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=FINE_TUNE_LABEL_SMOOTHING)
    optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=FINE_TUNE_LR, weight_decay=FINE_TUNE_WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    best_score = -1.0
    best_state = None
    patience_counter = 0
    use_amp = DEVICE.type == 'cuda'
    scaler_amp = torch.cuda.amp.GradScaler(enabled=use_amp)

    start_epoch = 0
    if FINE_TUNED_BACKBONE_CKPT_PATH.exists() and not FORCE_REFIT_BACKBONE:
        print('Memuat checkpoint fine-tune CNN:', FINE_TUNED_BACKBONE_CKPT_PATH, flush=True)
        ckpt = torch.load(FINE_TUNED_BACKBONE_CKPT_PATH, map_location='cpu')
        unwrap_model(model).load_state_dict(strip_module_prefix(ckpt['model_state']))
        if 'optimizer_state' in ckpt:
            try:
                optimizer.load_state_dict(ckpt['optimizer_state'])
                for state in optimizer.state.values():
                    for key, value in state.items():
                        if torch.is_tensor(value):
                            state[key] = value.to(DEVICE)
            except ValueError as err:
                print('Optimizer checkpoint tidak cocok dengan strategi fine-tune baru; optimizer di-reset:', err, flush=True)
        if 'scheduler_state' in ckpt:
            try:
                scheduler.load_state_dict(ckpt['scheduler_state'])
            except ValueError as err:
                print('Scheduler checkpoint tidak cocok; scheduler di-reset:', err, flush=True)
        best_score = float(ckpt.get('best_score', best_score))
        patience_counter = int(ckpt.get('patience_counter', patience_counter))
        best_state = ckpt.get('best_model_state')
        start_epoch = int(ckpt.get('epoch', 0))
        print(f'Resume fine-tune dari epoch {start_epoch + 1}/{FINE_TUNE_EPOCHS}', flush=True)

    for epoch in range(start_epoch, FINE_TUNE_EPOCHS):
        print(f'Mulai fine-tune CNN epoch {epoch+1}/{FINE_TUNE_EPOCHS}', flush=True)
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for batch_idx, (batch_x, batch_y) in enumerate(tqdm(train_loader, desc=f'Fine-tune CNN epoch {epoch+1}/{FINE_TUNE_EPOCHS}', unit='batch', dynamic_ncols=True), start=1):
            batch_x = batch_x.to(DEVICE, non_blocking=PIN_MEMORY)
            batch_y = batch_y.to(DEVICE, non_blocking=PIN_MEMORY)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=('cuda' if use_amp else 'cpu'), dtype=torch.float16, enabled=use_amp):
                logits = model(batch_x)
                loss = criterion(logits, batch_y)
            scaler_amp.scale(loss).backward()
            scaler_amp.step(optimizer)
            scaler_amp.update()

            train_loss += loss.item() * batch_x.size(0)
            train_correct += (torch.argmax(logits, dim=1) == batch_y).sum().item()
            train_total += batch_x.size(0)
            if batch_idx == 1 or batch_idx % 10 == 0 or batch_idx == len(train_loader):
                print(f'  batch {batch_idx}/{len(train_loader)} selesai', flush=True)

        print(f'Mulai validasi epoch {epoch+1}/{FINE_TUNE_EPOCHS}', flush=True)
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(DEVICE, non_blocking=PIN_MEMORY)
                with torch.autocast(device_type=('cuda' if use_amp else 'cpu'), dtype=torch.float16, enabled=use_amp):
                    logits = model(batch_x)
                val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
                val_true.extend(batch_y.numpy().tolist())

        train_acc = train_correct / train_total
        val_acc = accuracy_score(val_true, val_preds)
        val_macro_f1 = f1_score(val_true, val_preds, average='macro')
        score = 0.6 * val_acc + 0.4 * val_macro_f1
        scheduler.step(score)

        print(f'Epoch {epoch+1:02d} | Train Loss {train_loss/train_total:.4f} | Train Acc {train_acc:.3f} | Val Acc {val_acc:.3f} | Val Macro F1 {val_macro_f1:.3f}', flush=True)

        if score > best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in unwrap_model(model).state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        current_state = {k: v.detach().cpu().clone() for k, v in unwrap_model(model).state_dict().items()}
        torch.save({
            'epoch': epoch + 1,
            'model_state': current_state,
            'best_model_state': best_state,
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'best_score': best_score,
            'patience_counter': patience_counter,
        }, FINE_TUNED_BACKBONE_CKPT_PATH)
        print('Checkpoint fine-tune CNN disimpan:', FINE_TUNED_BACKBONE_CKPT_PATH, flush=True)

        if patience_counter >= FINE_TUNE_PATIENCE:
            print(f'Early stopping CNN fine-tune at epoch {epoch+1}', flush=True)
            break

    if best_state is not None:
        unwrap_model(model).load_state_dict(best_state)

    clean_state = unwrap_model(model).state_dict()
    torch.save(clean_state, FINE_TUNED_BACKBONE_PATH)
    print('Fine-tuned CNN disimpan ke:', FINE_TUNED_BACKBONE_PATH, flush=True)
    return unwrap_model(model).to(DEVICE), ft_preprocess


if TRAIN_SUPERVISED_BACKBONE:
    fine_tuned_classifier, preprocess = train_supervised_backbone()
    cnn_feature_extractor = feature_extractor_from_classifier(fine_tuned_classifier, BACKBONE_NAME).to(DEVICE)
    cnn_feature_extractor = maybe_wrap_multi_gpu(cnn_feature_extractor, name=f'{BACKBONE_NAME} fine-tuned feature extractor')
    cnn_feature_extractor.eval()
    print('Menggunakan fine-tuned CNN embedding untuk ANFIS.')
else:
    print('Menggunakan frozen pretrained CNN embedding untuk ANFIS.')


Menggunakan DataParallel untuk efficientnet_b0 classifier fine-tune: GPU [0, 1]
Trainable CNN params: 3,702,731/4,011,391
Auto-tune batch fine-tune CNN. Kandidat: [512, 384, 256, 192, 128, 96, 64]


Batch fine-tune CNN dipilih: 512
Batch fine-tune efektif: 512


Fine-tune CNN epoch 1/18:   0%|          | 0/37 [00:00<?, ?batch/s]

Epoch 01 | Train Loss 0.8961 | Train Acc 0.581 | Val Acc 0.744 | Val Macro F1 0.689


Fine-tune CNN epoch 2/18:   0%|          | 0/37 [00:00<?, ?batch/s]

...

Fine-tune CNN epoch 3/18:   0%|          | 0/37 [00:00<?, ?batch/s]

Epoch 03 | Train Loss 0.6300 | Train Acc 0.794 | Val Acc 0.814 | Val Macro F1 0.759


Fine-tune CNN epoch 4/18:   0%|          | 0/37 [00:00<?, ?batch/s]

Epoch 04 | Train Loss 0.5998 | Train Acc 0.815 | Val Acc 0.816 | Val Macro F1 0.764


Fine-tune CNN epoch 5/18:   0%|          | 0/37 [00:00<?, ?batch/s]

## 5. Ekstraksi CNN Embedding GPU

Embedding cache disimpan sebagai `.npy`. Ini membuat eksperimen PCA/rules bisa diulang tanpa ekstraksi CNN ulang.


In [ ]:
class GridImageDataset(Dataset):
    def __init__(self, df, image_dir, transform, augmentations):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.augmentations = augmentations
        self.image_cache = {} if NUM_WORKERS == 0 else None

    def __len__(self):
        return len(self.df)

    def _load_image(self, image_file):
        img_path = self.image_dir / image_file
        if self.image_cache is not None:
            if image_file not in self.image_cache:
                img = cv2.imread(str(img_path))
                if img is None:
                    raise FileNotFoundError(f'Gambar tidak bisa dibaca: {img_path}')
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                self.image_cache[image_file] = img
            return self.image_cache[image_file]

        img = cv2.imread(str(img_path))
        if img is None:
            raise FileNotFoundError(f'Gambar tidak bisa dibaca: {img_path}')
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    def _augment(self, pil_img, aug_name):
        if aug_name == 'original':
            return pil_img
        if aug_name == 'hflip':
            return ImageOps.mirror(pil_img)
        if aug_name == 'vflip':
            return ImageOps.flip(pil_img)
        raise ValueError(f'Augmentasi tidak dikenal: {aug_name}')

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load_image(row['image_file'])
        grid = img[row['y']:row['y'] + GRID_SIZE, row['x']:row['x'] + GRID_SIZE]

        if grid.shape[:2] != (GRID_SIZE, GRID_SIZE):
            raise ValueError(f'Grid tidak valid pada {row["Grid_ID"]}: {grid.shape}')

        pil_grid = Image.fromarray(grid)
        if self.augmentations == ['original']:
            # Jalur cepat: tidak perlu membuat tensor dimensi augmentasi lalu di-average.
            return self.transform(pil_grid), int(row['Y_Target']), row['Grid_ID']

        tensors = [self.transform(self._augment(pil_grid, aug)) for aug in self.augmentations]
        return torch.stack(tensors, dim=0), int(row['Y_Target']), row['Grid_ID']


def extract_cnn_embeddings(df, image_dir, transform, model, device):
    global BATCH_SIZE_EMBEDDING
    BATCH_SIZE_EMBEDDING = tune_cnn_batch_size(
        model,
        BATCH_SIZE_EMBEDDING_CANDIDATES,
        train_mode=False,
        name='ekstraksi embedding CNN',
    )
    print('Batch embedding efektif:', BATCH_SIZE_EMBEDDING)

    dataset = GridImageDataset(df, image_dir, transform, EMBEDDING_AUGMENTATIONS)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE_EMBEDDING,
        shuffle=False,
        **fast_loader_kwargs(),
    )

    all_embeddings, all_labels, all_grid_ids = [], [], []
    total_grids = len(dataset)
    total_batches = len(loader)
    processed_grids = 0
    use_amp = device.type == 'cuda'

    print(
        f'Mulai ekstraksi CNN embedding: {total_grids:,} grid | {total_batches:,} batch | '
        f'aug={len(EMBEDDING_AUGMENTATIONS)} | device={device} | multi_gpu={USE_MULTI_GPU}'
    )

    model.eval()
    with torch.no_grad():
        progress = tqdm(loader, total=total_batches, desc=f'Extracting {BACKBONE_NAME}', unit='batch', dynamic_ncols=True)
        for batch_imgs, batch_labels, batch_grid_ids in progress:
            if batch_imgs.ndim == 5:
                bsz, n_aug = batch_imgs.shape[:2]
                batch_imgs = batch_imgs.reshape(bsz * n_aug, *batch_imgs.shape[2:]).to(device, non_blocking=PIN_MEMORY)
                with torch.autocast(device_type=('cuda' if use_amp else 'cpu'), dtype=torch.float16, enabled=use_amp):
                    feats = model(batch_imgs)
                feats = feats.float().view(bsz, n_aug, -1).mean(dim=1)
            else:
                bsz = batch_imgs.size(0)
                batch_imgs = batch_imgs.to(device, non_blocking=PIN_MEMORY)
                with torch.autocast(device_type=('cuda' if use_amp else 'cpu'), dtype=torch.float16, enabled=use_amp):
                    feats = model(batch_imgs)
                feats = feats.float()

            all_embeddings.append(feats.cpu().numpy())
            all_labels.extend(batch_labels.numpy().tolist())
            all_grid_ids.extend(list(batch_grid_ids))

            processed_grids += bsz
            progress.set_postfix({'grid': f'{processed_grids:,}/{total_grids:,}', 'last_batch': bsz})

    embeddings = np.vstack(all_embeddings).astype(np.float32)
    labels = np.array(all_labels, dtype=np.int64)
    return embeddings, labels, all_grid_ids


def load_embedding_cache():
    if not (EMBEDDING_NPY.exists() and EMBEDDING_META_CSV.exists()):
        return None
    old_meta = pd.read_csv(EMBEDDING_META_CSV)
    if set(old_meta['Grid_ID']) != set(df_grid['Grid_ID']):
        print('Cache embedding tidak cocok dengan grid metadata saat ini. Ekstraksi ulang diperlukan.')
        return None
    emb = np.load(EMBEDDING_NPY)
    expected = df_grid['Grid_ID'].tolist()
    if old_meta['Grid_ID'].tolist() != expected:
        pos = {gid: i for i, gid in enumerate(old_meta['Grid_ID'])}
        emb = emb[[pos[gid] for gid in expected]]
    meta = df_grid[['Grid_ID', 'image_file', 'Y_Target']].copy()
    return emb, meta


cache = None if FORCE_REEXTRACT_EMBEDDING else load_embedding_cache()
if cache is not None:
    embeddings, embedding_meta = cache
    labels = embedding_meta['Y_Target'].values.astype(np.int64)
    print('Memuat embedding cache:', EMBEDDING_NPY)
else:
    embeddings, labels, grid_ids = extract_cnn_embeddings(df_grid, IMAGE_DIR, preprocess, cnn_feature_extractor, DEVICE)
    embedding_meta = df_grid[['Grid_ID', 'image_file', 'Y_Target']].copy()
    embedding_meta.to_csv(EMBEDDING_META_CSV, index=False)
    np.save(EMBEDDING_NPY, embeddings)
    print('Embedding disimpan ke:', EMBEDDING_NPY)
    print('Metadata embedding disimpan ke:', EMBEDDING_META_CSV)

print('Embedding shape:', embeddings.shape)
print('Label shape:', labels.shape)


## 6. Definisi ANFIS

Struktur ANFIS tetap dipakai sebagai classifier akhir. Tidak ada penggantian menjadi classifier deep learning.


In [ ]:
class ANFIS(nn.Module):
    def __init__(self, num_inputs, num_rules, num_classes):
        super().__init__()
        self.num_inputs = num_inputs
        self.num_rules = num_rules
        self.num_classes = num_classes

        self.mu = nn.Parameter(torch.randn(num_rules, num_inputs))
        self.sigma_raw = nn.Parameter(torch.ones(num_rules, num_inputs) * 0.5)

        self.consequent_weights = nn.Parameter(torch.randn(num_rules, num_inputs) * 0.05)
        self.consequent_bias = nn.Parameter(torch.zeros(num_rules))
        self.classifier = nn.Linear(num_rules, num_classes)

    @property
    def sigma(self):
        return F.softplus(self.sigma_raw) + 1e-6

    def init_from_kmeans(self, X_train_np):
        kmeans = KMeans(n_clusters=self.num_rules, random_state=SEED, n_init=10)
        kmeans.fit(X_train_np)

        self.mu.data = torch.tensor(kmeans.cluster_centers_, dtype=torch.float32)

        labels = kmeans.labels_
        sigma_init = torch.ones(self.num_rules, self.num_inputs) * 0.25
        for i in range(self.num_rules):
            cluster_data = X_train_np[labels == i]
            if len(cluster_data) > 1:
                std_vals = np.std(cluster_data, axis=0)
                std_vals = np.maximum(std_vals, 0.03)
                sigma_init[i] = torch.tensor(std_vals, dtype=torch.float32)

        self.sigma_raw.data = torch.log(torch.exp(sigma_init) - 1 + 1e-6)
        print(f"MF diinisialisasi dari KMeans ({self.num_rules} rules)")

    def forward(self, x):
        x_expanded = x.unsqueeze(1).expand(-1, self.num_rules, -1)
        sigma = self.sigma

        log_mf = -0.5 * ((x_expanded - self.mu) / sigma) ** 2
        log_firing = torch.sum(log_mf, dim=2)
        normalized_firing = torch.softmax(log_firing, dim=1)

        consequent = torch.sum(x_expanded * self.consequent_weights, dim=2) + self.consequent_bias
        weighted_output = normalized_firing * consequent
        logits = self.classifier(weighted_output)

        return logits, normalized_firing


## 7. Helper PCA, Training, dan Evaluasi


In [ ]:
def build_pca_features(embeddings, pca_components):
    pca = PCA(n_components=pca_components, random_state=SEED)
    pca.fit(embeddings[train_idx])
    pca_features = pca.transform(embeddings)

    scaler = MinMaxScaler()
    scaler.fit(pca_features[train_idx])
    X_all = scaler.transform(pca_features).astype(np.float32)
    feature_cols = [f'CNN_PCA_{i+1:02d}' for i in range(pca_components)]
    return X_all, pca, scaler, feature_cols


def split_arrays(X_all, y_all):
    return (
        X_all[train_idx], y_all[train_idx],
        X_all[val_idx], y_all[val_idx],
        X_all[test_idx], y_all[test_idx],
    )


def make_class_weights(y_train, power=1.0):
    counts = np.bincount(y_train, minlength=NUM_CLASSES)
    weights = np.array([len(y_train) / (NUM_CLASSES * max(c, 1)) for c in counts], dtype=np.float32)
    weights = weights ** power
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


def train_anfis_model(config, X_train, y_train, X_val, y_val, epochs, patience, verbose=False):
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32, device=DEVICE)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32, device=DEVICE)
    y_val_tensor = torch.tensor(y_val, dtype=torch.long, device=DEVICE)

    model = ANFIS(X_train.shape[1], int(config['num_rules']), NUM_CLASSES)
    model.init_from_kmeans(X_train)
    model.to(DEVICE)

    criterion = nn.CrossEntropyLoss(
        weight=make_class_weights(y_train, config.get('class_weight_power', 1.0)),
        label_smoothing=float(config.get('label_smoothing', 0.0)),
    )
    optimizer = optim.Adam(
        model.parameters(),
        lr=float(config['learning_rate']),
        weight_decay=float(config.get('weight_decay', 0.0)),
    )
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=8)

    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=min(BATCH_SIZE_TRAIN, len(X_train_tensor)), shuffle=True)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        for batch_X, batch_y in train_loader:
            logits, _ = model(batch_X)
            loss = criterion(logits, batch_y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * batch_X.size(0)
            total_correct += (torch.argmax(logits, dim=1) == batch_y).sum().item()
            total_samples += batch_X.size(0)

        train_loss = total_loss / total_samples
        train_acc = total_correct / total_samples

        model.eval()
        with torch.no_grad():
            val_logits, _ = model(X_val_tensor)
            val_loss = criterion(val_logits, y_val_tensor).item()
            val_pred = torch.argmax(val_logits, dim=1)
            val_acc = (val_pred == y_val_tensor).float().mean().item()

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if verbose and ((epoch + 1) % 25 == 0 or patience_counter == 0):
            print(f'Epoch {epoch+1:03d} | Train Loss {train_loss:.4f} | Val Loss {val_loss:.4f} | Val Acc {val_acc:.3f}')

        if patience_counter >= patience:
            if verbose:
                print(f'Early stopping at epoch {epoch+1}')
            break

    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(DEVICE)

    return model, history, best_val_loss


def predict_anfis(model, X):
    model.eval()
    X_tensor = torch.tensor(X, dtype=torch.float32, device=DEVICE)
    preds = []
    with torch.no_grad():
        for start in range(0, len(X_tensor), PREDICT_BATCH_SIZE):
            logits, _ = model(X_tensor[start:start+PREDICT_BATCH_SIZE])
            preds.append(torch.argmax(logits, dim=1).cpu().numpy())
    return np.concatenate(preds)


def evaluate_config(config, embeddings, labels, epochs=SEARCH_EPOCHS, patience=SEARCH_PATIENCE, verbose=False):
    X_all, pca, scaler, feature_cols = build_pca_features(embeddings, int(config['pca_components']))
    X_train, y_train, X_val, y_val, X_test, y_test = split_arrays(X_all, labels)
    model, history, best_val_loss = train_anfis_model(config, X_train, y_train, X_val, y_val, epochs, patience, verbose=verbose)
    val_pred = predict_anfis(model, X_val)
    val_acc = accuracy_score(y_val, val_pred)
    val_macro_f1 = f1_score(y_val, val_pred, average='macro')
    score = 0.65 * val_acc + 0.35 * val_macro_f1
    return {'score': score, 'val_acc': val_acc, 'val_macro_f1': val_macro_f1, 'best_val_loss': best_val_loss, 'config': copy.deepcopy(config)}


## 8. Genetic-Style Soft Search untuk PCA + ANFIS

Search ini memakai konsep evolusioner sederhana: populasi, elitisme, crossover, dan mutasi. ANFIS tetap menjadi model klasifikasi akhir.


In [ ]:
def random_config():
    return {key: random.choice(values) for key, values in SEARCH_SPACE.items()}


def config_key(config):
    return tuple((k, config[k]) for k in sorted(config.keys()))


def crossover(parent_a, parent_b):
    return {key: (parent_a[key] if random.random() < 0.5 else parent_b[key]) for key in SEARCH_SPACE}


def mutate(config):
    child = copy.deepcopy(config)
    for key, values in SEARCH_SPACE.items():
        if random.random() < GA_MUTATION_RATE:
            child[key] = random.choice(values)
    return child


def run_ga_search():
    evaluated = {}
    rows = []
    population = [random_config() for _ in range(GA_POPULATION - 1)] + [copy.deepcopy(FALLBACK_CONFIG)]
    best_result = None

    for gen in range(GA_GENERATIONS):
        print(f'\n=== GA Generation {gen+1}/{GA_GENERATIONS} ===')
        gen_results = []

        for config in tqdm(population, desc='Evaluating ANFIS configs', unit='config', dynamic_ncols=True):
            key = config_key(config)
            if key not in evaluated:
                result = evaluate_config(config, embeddings, labels)
                evaluated[key] = result
                row = copy.deepcopy(config)
                row.update({'generation': gen + 1, 'score': result['score'], 'val_acc': result['val_acc'], 'val_macro_f1': result['val_macro_f1'], 'best_val_loss': result['best_val_loss']})
                rows.append(row)
            result = evaluated[key]
            gen_results.append(result)

            if best_result is None or result['score'] > best_result['score']:
                best_result = result
                print('Best sementara:', {**result['config'], 'score': round(result['score'], 4), 'val_acc': round(result['val_acc'], 4), 'macro_f1': round(result['val_macro_f1'], 4)})

        gen_results = sorted(gen_results, key=lambda r: r['score'], reverse=True)
        elites = [r['config'] for r in gen_results[:max(2, GA_POPULATION // 4)]]
        next_population = [copy.deepcopy(cfg) for cfg in elites]
        while len(next_population) < GA_POPULATION:
            pa, pb = random.sample(elites, 2)
            next_population.append(mutate(crossover(pa, pb)))
        population = next_population

    df_results = pd.DataFrame(rows).sort_values('score', ascending=False).reset_index(drop=True)
    df_results.to_csv(SEARCH_RESULTS_CSV, index=False)
    print('Search selesai. Hasil disimpan ke:', SEARCH_RESULTS_CSV)
    return best_result, df_results


if RUN_GA_SEARCH:
    best_search_result, df_search_results = run_ga_search()
    best_config = best_search_result['config']
else:
    best_search_result = evaluate_config(FALLBACK_CONFIG, embeddings, labels, verbose=True)
    best_config = best_search_result['config']
    df_search_results = pd.DataFrame([{**best_config, 'score': best_search_result['score'], 'val_acc': best_search_result['val_acc'], 'val_macro_f1': best_search_result['val_macro_f1']}])

print('Best config:', best_config)
df_search_results.head(10)


## 9. Final Training ANFIS dengan Konfigurasi Terbaik


In [ ]:
X_all, pca, scaler, feature_cols = build_pca_features(embeddings, int(best_config['pca_components']))
X_train, y_train, X_val, y_val, X_test, y_test = split_arrays(X_all, labels)

print('Final feature shape:', X_all.shape)
print('Train:', X_train.shape, 'Val:', X_val.shape, 'Test:', X_test.shape)
print('Best config:', best_config)
print('PCA explained variance total:', float(np.sum(pca.explained_variance_ratio_)))

anfis_model, final_history, best_val_loss = train_anfis_model(best_config, X_train, y_train, X_val, y_val, FINAL_EPOCHS, FINAL_PATIENCE, verbose=True)


## 10. Evaluasi Test Set


In [ ]:
y_pred = predict_anfis(anfis_model, X_test)
y_true = y_test

test_accuracy = accuracy_score(y_true, y_pred)
test_macro_f1 = f1_score(y_true, y_pred, average='macro')

print(f'Test Accuracy: {test_accuracy * 100:.2f}%')
print(f'Test Macro F1: {test_macro_f1:.4f}')
print(classification_report(y_true, y_pred, target_names=['Aman', 'Tersebar', 'Kritis']))

cm = confusion_matrix(y_true, y_pred)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(final_history['train_loss'], label='Train Loss')
axes[0].plot(final_history['val_loss'], label='Val Loss')
axes[0].set_title('Loss'); axes[0].grid(alpha=0.3); axes[0].legend()
axes[1].plot(final_history['train_acc'], label='Train Acc')
axes[1].plot(final_history['val_acc'], label='Val Acc')
axes[1].set_title('Accuracy'); axes[1].grid(alpha=0.3); axes[1].legend()
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Aman', 'Tersebar', 'Kritis'], yticklabels=['Aman', 'Tersebar', 'Kritis'], ax=axes[2])
axes[2].set_title('Confusion Matrix'); axes[2].set_xlabel('Prediksi'); axes[2].set_ylabel('Aktual')
plt.tight_layout(); plt.show()


## 11. Simpan Artefak Final

Hanya artefak final yang disimpan. Trial GA tidak menyimpan model satu per satu.


## 12. Catatan Optimasi Versi 2 GPU T4 Max Batch

Implementasi pada versi ini diubah agar lebih agresif memakai resource Kaggle **GPU T4 x2** dan mengurangi waktu eksekusi:

```text
1. EfficientNet-B0 pretrained sebagai backbone default.
2. Supervised fine-tuning CNN pada label grid agar embedding spesifik terhadap sampah UAV.
3. CNN fine-tuning dan ekstraksi embedding dibungkus dengan torch.nn.DataParallel jika terdeteksi >1 GPU.
4. Embedding hanya memakai original grid, tanpa hflip/vflip, sehingga forward pass CNN turun dari 3x menjadi 1x.
5. Fine-tune hemat GPU: maksimal 5 epoch, patience 2, hanya classifier + 1 blok terakhir yang dilatih.
6. Batch fine-tune tetap auto-tune, target awal 512 global pada T4 x2 jika memori aman.
7. Batch embedding dinaikkan agresif: target awal 2048 global pada T4 x2.
8. GA search dipangkas menjadi 4 populasi x 2 generasi agar muat dalam sisa kuota GPU.
9. DataLoader dibuat konservatif: NUM_WORKERS maksimal 2, persistent_workers=False, dan timeout aktif agar hang tidak silent.
10. Fine-tune CNN menyimpan checkpoint per epoch dan bisa resume dari checkpoint tersebut.
11. Mixed precision tetap aktif saat fine-tuning dan ekstraksi embedding di GPU T4.
12. PCA + MinMax scaling + ANFIS tetap menjadi pipeline klasifikasi final.
```

Catatan penting: DataParallel mempercepat bagian CNN, terutama fine-tuning dan ekstraksi embedding. Bagian PCA, GA search, dan ANFIS final tetap dominan berjalan di CPU/cuda:0 karena ukurannya lebih kecil dan tidak otomatis terbagi ke 2 GPU.

Jika Kaggle menampilkan VRAM belum penuh tetapi GPU utilization sudah tinggi, itu tetap normal. Target utama bukan memenuhi VRAM, melainkan menurunkan waktu per epoch/batch.


In [ ]:
df_cnn = pd.DataFrame(X_all, columns=feature_cols)
df_cnn.insert(0, 'Grid_ID', df_grid['Grid_ID'].values)
df_cnn['Y_Target'] = labels
df_cnn['image_file'] = df_grid['image_file'].values
df_cnn.to_csv(CNN_FEATURE_CSV, index=False)

joblib.dump(pca, PCA_PATH)
joblib.dump(scaler, SCALER_PATH)

save_dict = {
    'model_state_dict': anfis_model.state_dict(),
    'num_inputs': len(feature_cols),
    'num_rules': int(best_config['num_rules']),
    'num_classes': NUM_CLASSES,
    'feature_columns': feature_cols,
    'best_config': best_config,
    'pca_path': str(PCA_PATH),
    'scaler_path': str(SCALER_PATH),
    'cnn_backbone': BACKBONE_NAME,
    'embedding_augmentations': EMBEDDING_AUGMENTATIONS,
    'embedding_dim': int(embeddings.shape[1]),
    'grid_size': GRID_SIZE,
    'best_val_loss': float(best_val_loss),
    'test_accuracy': float(test_accuracy),
    'test_macro_f1': float(test_macro_f1),
}

torch.save(save_dict, MODEL_PATH)
print('Fitur PCA final:', CNN_FEATURE_CSV)
print('PCA final:', PCA_PATH)
print('Scaler final:', SCALER_PATH)
print('Model final:', MODEL_PATH)
